# 5. Benchmark Comparison - Tong hop va Phan tich

## Muc tieu
So sanh toan dien 3 models da cai dat:
1. **Faster R-CNN** (ResNet-50 + FPN) - Two-stage detector
2. **YOLOv3** (Darknet-53) - One-stage, anchor-based
3. **YOLOv8m** (CSPDarknet) - One-stage, anchor-free

Cac truc so sanh:
- Do chinh xac (mAP@0.5, mAP@0.5:0.95)
- Toc do (FPS, inference time)
- Model size (parameters)
- Per-class performance
- Speed vs Accuracy trade-off

In [ ]:
import sys
sys.path.append("../src")

import os
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from dataset import VOC_CLASSES
from visualize import (
    plot_benchmark_comparison, plot_per_class_comparison,
    plot_model_evolution_timeline
)

plt.style.use("seaborn-v0_8-whitegrid")
%matplotlib inline

RESULTS_DIR = "../results"

## 5.1 Load ket qua tu cac notebooks truoc

In [ ]:
# Load ket qua da luu tu cac notebooks truoc
def load_results(model_name):
    path = os.path.join(RESULTS_DIR, model_name, "eval_results.json")
    with open(path, "r") as f:
        return json.load(f)

faster_rcnn = load_results("faster_rcnn")
yolov3 = load_results("yolov3")
yolov8 = load_results("yolov8")

all_results = {
    "Faster R-CNN": faster_rcnn,
    "YOLOv3": yolov3,
    "YOLOv8m": yolov8,
}

# In bang tong hop
print(f"{'='*70}")
print(f"  {'Model':<15} {'mAP@0.5':>10} {'mAP@0.5:95':>12} {'FPS':>8} {'Params':>12} {'ms/img':>10}")
print(f"{'='*70}")
for name, r in all_results.items():
    params = r.get("total_params", 0)
    params_str = f"{params/1e6:.1f}M" if params else "N/A"
    print(f"  {name:<15} {r['mAP_50']:>10.4f} {r['mAP_50_95']:>12.4f} {r.get('fps', 0):>8.1f} {params_str:>12} {r.get('avg_inference_ms', 0):>10.1f}")
print(f"{'='*70}")

## 5.2 Timeline su phat trien Object Detection

In [ ]:
# Timeline - Su phat trien cua object detection
fig = plot_model_evolution_timeline()
plt.savefig(os.path.join(RESULTS_DIR, "evolution_timeline.png"), dpi=150, bbox_inches="tight")
plt.show()

## 5.3 So sanh tong the: mAP, FPS, Speed-Accuracy

In [ ]:
# Bieu do so sanh tong the
fig = plot_benchmark_comparison(all_results)
plt.savefig(os.path.join(RESULTS_DIR, "benchmark_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

## 5.4 So sanh Per-Class AP

In [ ]:
# Per-class comparison
fig = plot_per_class_comparison(all_results)
plt.savefig(os.path.join(RESULTS_DIR, "per_class_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

## 5.5 Phan tich chi tiet: Model nao tot hon o dau?

In [ ]:
# Phan tich: Tim classes ma moi model tot nhat / kem nhat
print("="*70)
print("  PHAN TICH: Model nao tot nhat cho tung class?")
print("="*70)

model_names = list(all_results.keys())

print(f"\n  {'Class':<15}", end="")
for name in model_names:
    print(f" {name:>14}", end="")
print(f" {'Best Model':>14}")
print(f"  {'-'*70}")

for cls in VOC_CLASSES:
    aps = []
    for name in model_names:
        per_class = all_results[name].get("per_class", {})
        if cls in per_class and isinstance(per_class[cls], dict):
            aps.append(per_class[cls].get("ap", 0))
        else:
            aps.append(0)
    
    best_idx = np.argmax(aps)
    print(f"  {cls:<15}", end="")
    for i, ap in enumerate(aps):
        marker = " *" if i == best_idx else "  "
        print(f" {ap:>12.4f}{marker}", end="")
    print(f" {model_names[best_idx]:>14}")

# Tong hop
print(f"\n{'='*70}")
print(f"  TONG HOP:")
print(f"{'='*70}")
for name in model_names:
    wins = 0
    for cls in VOC_CLASSES:
        aps = []
        for n in model_names:
            pc = all_results[n].get("per_class", {})
            if cls in pc and isinstance(pc[cls], dict):
                aps.append(pc[cls].get("ap", 0))
            else:
                aps.append(0)
        if model_names[np.argmax(aps)] == name:
            wins += 1
    print(f"  {name}: tot nhat o {wins}/{len(VOC_CLASSES)} classes")

## 5.6 Efficiency Analysis - Parameters vs Performance

In [ ]:
# Efficiency: mAP per million parameters
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

model_names = list(all_results.keys())
colors = ["#2196F3", "#FF9800", "#4CAF50"]

# 1. Params vs mAP
ax = axes[0]
for i, (name, r) in enumerate(all_results.items()):
    params_m = r.get("total_params", 0) / 1e6
    map50 = r["mAP_50"]
    ax.scatter(params_m, map50, s=300, c=colors[i], edgecolors="black", zorder=5)
    ax.annotate(name, (params_m, map50), textcoords="offset points",
                xytext=(10, 10), fontsize=11)

ax.set_xlabel("Parameters (Millions)", fontsize=12)
ax.set_ylabel("mAP@0.5", fontsize=12)
ax.set_title("Model Size vs Accuracy")
ax.grid(True, alpha=0.3)

# 2. Efficiency score = mAP / params(M)
ax = axes[1]
efficiencies = []
for name, r in all_results.items():
    params_m = r.get("total_params", 1) / 1e6
    eff = r["mAP_50"] / params_m if params_m > 0 else 0
    efficiencies.append(eff)

bars = ax.bar(model_names, efficiencies, color=colors, edgecolor="black")
ax.set_ylabel("mAP@0.5 / Million Params")
ax.set_title("Parameter Efficiency")
for bar, eff in zip(bars, efficiencies):
    ax.text(bar.get_x() + bar.get_width()/2, eff + 0.001,
            f"{eff:.4f}", ha="center", fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "efficiency_analysis.png"), dpi=150, bbox_inches="tight")
plt.show()

## 5.7 Ket luan tong hop

### 1. Su phat trien cua Object Detection

| Giai doan | Model | Van de giai quyet | Trade-off |
|-----------|-------|-------------------|-----------|
| **2014** | R-CNN | CNN cho detection | Cuc cham (47s/img) |
| **2015** | Fast R-CNN | Shared features, RoI Pooling | Van dung Selective Search |
| **2015** | Faster R-CNN | RPN thay Selective Search | Van 2 stages, ~5 FPS |
| **2016** | YOLOv1 | One-stage, real-time | Mat accuracy, small objects kem |
| **2018** | YOLOv3 | Multi-scale, Darknet-53 | Balance speed-accuracy |
| **2023** | YOLOv8 | Anchor-free, Decoupled Head | State-of-the-art |

### 2. Key Insights tu ket qua thuc nghiem

**Two-Stage vs One-Stage:**
- Faster R-CNN cho bbox chinh xac hon (RoI Align refine bbox)
- YOLO nhanh hon nhieu, phu hop real-time applications
- Gap ve accuracy da thu hep dang ke: YOLOv8 gan ngang Faster R-CNN ve mAP

**Anchor-based vs Anchor-free:**
- YOLOv3 (anchor-based): Can chon anchors phu hop, sensitive voi dataset
- YOLOv8 (anchor-free): Linh hoat hon, it hyperparameters, generalize tot hon

**Model choice phu thuoc vao use case:**
- Can **do chinh xac cao** (medical imaging, quality control): Faster R-CNN
- Can **real-time** (self-driving, surveillance): YOLOv8
- Can **can bang** (robotics, mobile): YOLOv8n/s (lighter variants)

### 3. Huong phat trien tuong lai
- **Transformer-based**: DETR, DINO - su dung attention mechanism
- **Anchor-free domination**: Xu huong loai bo anchor boxes
- **Neural Architecture Search**: Tu dong thiet ke kien truc
- **Edge deployment**: Model nho, nhanh cho mobile/embedded devices

### 4. Han che cua do an nay
- Chi fine-tune tu pretrained, khong train tu scratch (do gioi han compute)
- Dataset VOC 2012 nho hon COCO -> ket qua co the khac tren dataset lon hon
- Chua so sanh voi cac model moi nhat (YOLOv9, RT-DETR)
- Inference speed phu thuoc hardware, ket qua khac nhau tren GPU khac nhau